# Lab 4 — RAG over RDF/SPARQL
**Course:** Web Mining & Semantics — ESILV  
**Domain:** Men's Volleyball (FIVB)  
**LLM:** Claude Haiku via Anthropic API  
**KB:** `expanded_clean.nt` — 84,016 triples (Lab 2)

We build a chatbot that answers natural language questions by generating SPARQL
from the KB, executing it with rdflib, and self-repairing if the query fails.


## 0. Setup

In [ ]:
!pip install -q rdflib anthropic pandas matplotlib SPARQLWrapper

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
ARTIFACTS_DIR = '/content/drive/MyDrive/volleyball-kg/kg_artifacts'
DATA_DIR      = '/content/drive/MyDrive/volleyball-kg/data'
RAG_DIR       = '/content/drive/MyDrive/volleyball-kg/rag'
os.makedirs(RAG_DIR, exist_ok=True)
print('Directories ready ✅')

In [ ]:
import re, json
import pandas as pd
import anthropic
from rdflib import Graph, Namespace
from SPARQLWrapper import SPARQLWrapper, JSON as SPARQL_JSON
from google.colab import userdata

# Anthropic client
client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))
MODEL  = 'claude-haiku-4-5-20251001'
MAX_REPAIR_ATTEMPTS = 2

print(f'Anthropic client ready ✅  ({MODEL})')

## 1. Load RDF Graph

In [ ]:
g = Graph()
g.parse(f'{ARTIFACTS_DIR}/expanded_clean.nt', format='nt')
print(f'Loaded {len(g):,} triples ✅')

WD  = Namespace('http://www.wikidata.org/entity/')
WDT = Namespace('http://www.wikidata.org/prop/direct/')
VB  = Namespace('http://volleyball-kg.org/ontology#')
g.bind('wd', WD); g.bind('wdt', WDT); g.bind('vb', VB)

## 2. Schema Summary

In [ ]:
PREFIXES = {
    'rdf':  'http://www.w3.org/1999/02/22-rdf-syntax-ns#',
    'rdfs': 'http://www.w3.org/2000/01/rdf-schema#',
    'owl':  'http://www.w3.org/2002/07/owl#',
    'xsd':  'http://www.w3.org/2001/XMLSchema#',
    'vb':   'http://volleyball-kg.org/ontology#',
    'wd':   'http://www.wikidata.org/entity/',
    'wdt':  'http://www.wikidata.org/prop/direct/',
}

ENTITY_HINTS = """# Key entity URIs
# Countries: wd:Q142=France  wd:Q36=Poland  wd:Q155=Brazil  wd:Q38=Italy
#            wd:Q30=UnitedStates  wd:Q241=Cuba  wd:Q46=Europe  wd:Q18=SouthAmerica
# Players:   wd:Q3046479=EarvinNgapeth(France,outsideHitter)
# Orgs:      wd:Q6851=FIVB  wd:Q1543502=CEV
# Predicates: wdt:P1532=representsCountry  wdt:P19=bornIn  wdt:P27=citizenship
#             wdt:P413=position  wdt:P17=country  wdt:P664=organizer
#             wdt:P156=followedBy  wdt:P31=instanceOf  wdt:P641=sport"""

def shorten(uri):
    for p, ns in PREFIXES.items():
        if str(uri).startswith(ns):
            return f'{p}:{str(uri)[len(ns):]}'
    return str(uri).split('/')[-1]

def build_schema_summary(g):
    prefix_block = '\n'.join(f'PREFIX {p}: <{ns}>' for p, ns in PREFIXES.items())
    preds   = [shorten(r.p)   for r in g.query('SELECT DISTINCT ?p WHERE { ?s ?p ?o . FILTER(isIRI(?o)) } LIMIT 60')]
    classes = [shorten(r.cls) for r in g.query('SELECT DISTINCT ?cls WHERE { ?s a ?cls . } LIMIT 30')]
    samples = [(shorten(r.s), shorten(r.p), shorten(r.o))
               for r in g.query('SELECT ?s ?p ?o WHERE { ?s ?p ?o . FILTER(isIRI(?o)) } LIMIT 15')]
    return (f"{prefix_block}\n\n{ENTITY_HINTS}\n\n"
            f"# Predicates\n" + '\n'.join(f'- {p}' for p in preds) +
            f"\n\n# Entity types\n" + '\n'.join(f'- {c}' for c in classes) +
            f"\n\n# Sample triples\n" + '\n'.join(f'- {s} {p} {o}' for s,p,o in samples))

schema_summary = build_schema_summary(g)
print(f'Schema summary built: {len(schema_summary):,} chars ✅')

## 3. LLM Functions & Label Resolver

In [ ]:
FEW_SHOT = """# WORKING SPARQL PATTERNS

## Players of a country:
SELECT ?player WHERE { ?player wdt:P1532 wd:Q142 . } LIMIT 20

## Position of a player:
SELECT ?pos WHERE { wd:Q3046479 wdt:P413 ?pos . }

## Players born in a country (via city):
SELECT ?player ?city WHERE { ?player wdt:P19 ?city . ?city wdt:P17 wd:Q38 . } LIMIT 20

## Players with a citizenship:
SELECT ?player WHERE { ?player wdt:P27 wd:Q241 . } LIMIT 20

## Clubs from a country:
SELECT ?club WHERE {
    ?club wdt:P17 wd:Q38 .
    ?club wdt:P31 ?type .
    FILTER(CONTAINS(LCASE(STR(?type)), "club") || CONTAINS(LCASE(STR(?type)), "sport"))
} LIMIT 20

## Tournaments organized by FIVB:
SELECT ?t WHERE { ?t wdt:P664 wd:Q6851 . } LIMIT 20

## Tournaments followed by another edition:
SELECT ?t1 ?t2 WHERE { ?t1 wdt:P156 ?t2 . } LIMIT 20

## Sport of an organization:
SELECT ?sport WHERE { wd:Q6851 wdt:P641 ?sport . }

## Teams from a continent:
SELECT ?team ?country WHERE { ?team wdt:P17 ?country . ?country wdt:P30 wd:Q46 . } LIMIT 20

## Players born AND representing the same country:
SELECT ?player WHERE {
    ?player wdt:P1532 wd:Q142 .
    ?player wdt:P19 ?city .
    ?city wdt:P17 wd:Q142 .
} LIMIT 20"""

SPARQL_PROMPT = """You are a SPARQL expert for a volleyball knowledge graph.
Convert the QUESTION into a SPARQL 1.1 SELECT query.
- ALWAYS use wdt:P1532 for country representation (never vb:representsCountry)
- Use the QIDs from the entity hints
- Follow the working patterns above
- Return ONLY the SPARQL inside ```sparql ... ```"""

REPAIR_PROMPT = """The SPARQL below failed or returned 0 results. Fix it.
- Use wdt:P1532 for country representation
- If searching by name: FILTER(CONTAINS(LCASE(STR(?x)), "keyword"))
- Return ONLY the corrected SPARQL inside ```sparql ... ```"""

CODE_RE = re.compile(r'```(?:sparql)?\s*(.*?)```', re.IGNORECASE | re.DOTALL)

def extract_sparql(text):
    m = CODE_RE.search(text)
    return m.group(1).strip() if m else text.strip()

def call_claude(system, user):
    msg = client.messages.create(
        model=MODEL, max_tokens=512, system=system,
        messages=[{'role': 'user', 'content': user}]
    )
    return msg.content[0].text

def generate_sparql(question):
    return extract_sparql(call_claude(SPARQL_PROMPT,
        f"{FEW_SHOT}\n\nSCHEMA SUMMARY:\n{schema_summary}\n\nQUESTION: {question}"))

def repair_sparql(question, bad_query, issue):
    return extract_sparql(call_claude(REPAIR_PROMPT,
        f"{FEW_SHOT}\n\nSCHEMA SUMMARY:\n{schema_summary}\n\nQUESTION: {question}\n\nBAD SPARQL:\n{bad_query}\n\nISSUE: {issue}"))

def run_sparql(query):
    res = g.query(query)
    return [str(v) for v in res.vars], [tuple(str(c) for c in r) for r in res]

def answer_baseline(question):
    return call_claude('You are a volleyball expert. Answer briefly in 2 sentences.', question)

# Label resolver — alignment table first, then Wikidata SPARQL for the rest
LABELS = {}
try:
    _df = pd.read_csv(f'{ARTIFACTS_DIR}/alignment_table.csv')
    for _, row in _df.iterrows():
        if pd.notna(row.get('wikidata_uri')) and pd.notna(row.get('label')):
            LABELS[str(row['wikidata_uri']).split('/')[-1]] = str(row['label'])
    print(f'Loaded {len(LABELS)} labels from alignment table')
except Exception as e:
    print(f'Alignment table not loaded: {e}')

def enrich_labels(qids):
    missing = [q for q in set(qids) if q.startswith('Q') and q not in LABELS]
    if not missing:
        return
    sparql_wd = SPARQLWrapper('https://query.wikidata.org/sparql')
    sparql_wd.setReturnFormat(SPARQL_JSON)
    sparql_wd.addCustomHttpHeader('User-Agent', 'VolleyballKG-Lab4/1.0')
    resolved = 0
    for i in range(0, len(missing), 80):
        batch  = missing[i:i+80]
        values = ' '.join(f'wd:{q}' for q in batch)
        try:
            sparql_wd.setQuery(f"""SELECT ?item ?label WHERE {{
              VALUES ?item {{ {values} }}
              ?item rdfs:label ?label . FILTER(LANG(?label)='en') }}""")
            for b in sparql_wd.query().convert()['results']['bindings']:
                LABELS[b['item']['value'].split('/')[-1]] = b['label']['value']
                resolved += 1
        except Exception:
            pass
    if resolved:
        print(f'Resolved {resolved}/{len(missing)} labels from Wikidata')

def readable(uri):
    qid = str(uri).split('/')[-1]
    if not qid.startswith('Q'):
        return qid.replace('_', ' ')
    return LABELS.get(qid, qid)

print('LLM functions ready ✅')
print(f'API test: {call_claude("Reply: OK", "test").strip()[:5]} ✅')

## 4. RAG Pipeline with Self-Repair

In [ ]:
def rag_answer(question):
    sparql  = generate_sparql(question)
    repairs = 0
    for attempt in range(MAX_REPAIR_ATTEMPTS + 1):
        try:
            vars_, rows = run_sparql(sparql)
            if not rows and attempt < MAX_REPAIR_ATTEMPTS:
                sparql = repair_sparql(question, sparql, '0 results returned')
                repairs += 1
                continue
            return {'sparql': sparql, 'vars': vars_, 'rows': rows, 'repairs': repairs, 'error': None}
        except Exception as e:
            if attempt < MAX_REPAIR_ATTEMPTS:
                sparql = repair_sparql(question, sparql, str(e))
                repairs += 1
            else:
                return {'sparql': sparql, 'vars': [], 'rows': [], 'repairs': repairs, 'error': str(e)}
    return {'sparql': sparql, 'vars': [], 'rows': [], 'repairs': repairs, 'error': 'Max repairs reached'}

print('RAG pipeline ready ✅')

## 5. Evaluation — 20 Questions

In [ ]:
EVALUATION_QUESTIONS = [
    'Which players represent France in volleyball?',
    'Which players represent Poland in volleyball?',
    'Which players represent Brazil in volleyball?',
    'Which players represent Italy in volleyball?',
    'Which players represent the United States in volleyball?',
    'What playing position does Earvin Ngapeth have?',
    'Which players were born in Italy?',
    'Which players are citizens of Cuba?',
    'Which players were born in France?',
    'Which players play as outside hitter?',
    'Which national teams are from European countries?',
    'Which national teams are from South American countries?',
    'What is the country of the French national volleyball team?',
    'Which clubs are from Italy?',
    'Which clubs are from Poland?',
    'What country is Trentino Volley associated with?',
    'Which tournaments are organized by the FIVB?',
    'Which tournaments are followed by another edition?',
    'What sport is the FIVB associated with?',
    'Which players both represent France and were born in France?',
]

# Baseline evaluated for 5 questions (lab requirement: at least 5)
BASELINE_IDS = {1, 6, 17, 18, 20}

print(f'Running evaluation on {len(EVALUATION_QUESTIONS)} questions...')
print()

eval_results = []

for i, question in enumerate(EVALUATION_QUESTIONS, 1):
    print(f'[{i:02d}/20] {question[:65]}')

    baseline = '—'
    if i in BASELINE_IDS:
        baseline = answer_baseline(question).strip()

    result = rag_answer(question)
    rows   = result['rows']
    n      = len(rows)
    ok     = n > 0

    eval_results.append({
        'id': i, 'question': question, 'baseline': baseline,
        'sparql': result['sparql'][:300], 'n_results': n,
        'correct': ok, 'repairs': result['repairs'],
        'error': result['error'] or '',
        '_rows': rows, '_vars': result['vars'],
    })

    tag = '✅' if ok else '❌'
    rep = f' [repaired {result["repairs"]}x]' if result['repairs'] else ''
    print(f'       {tag}  {n} results{rep}')

print()
n_ok  = sum(1 for r in eval_results if r['correct'])
n_rep = sum(1 for r in eval_results if r['repairs'] > 0)
print(f'Success rate : {n_ok}/20 ({n_ok/20*100:.0f}%)')
print(f'Self-repairs : {n_rep}/20')

## 6. Results Table

In [ ]:
from IPython.display import display

# Resolve all QIDs to readable names in one batch
print('Fetching labels from Wikidata...')
all_qids = list(set(
    str(c).split('/')[-1]
    for r in eval_results
    for row in r['_rows']
    for c in row
    if str(c).split('/')[-1].startswith('Q')
))
enrich_labels(all_qids)
print()

# Build results dataframe
rows_table = []
for r in eval_results:
    answers = [readable(row[0]) for row in r['_rows']]
    rows_table.append({
        '#':        r['id'],
        'Question': r['question'],
        'Results':  r['n_results'],
        'Answers':  ', '.join(answers) if answers else '—',
        'Repaired': f'{r["repairs"]}x' if r['repairs'] else 'no',
        'Correct':  'YES ✅' if r['correct'] else 'NO ❌',
    })

df = pd.DataFrame(rows_table)
display(df)

df.to_csv(f'{RAG_DIR}/evaluation_table.csv', index=False)
print(f'Table saved → {RAG_DIR}/evaluation_table.csv')

# Baseline vs RAG for the 5 selected questions
print()
print('Baseline vs RAG (5 selected questions):')
print('='*80)
for r in eval_results:
    if r['id'] in BASELINE_IDS:
        answers = [readable(row[0]) for row in r['_rows']]
        print(f'\nQ{r["id"]:02d}: {r["question"]}')
        print(f'  Baseline : {r["baseline"][:200]}')
        print(f'  RAG      : {", ".join(answers) if answers else "(no results)"}')
        print(f'  Correct  : {"YES ✅" if r["correct"] else "NO ❌"}')

## 7. Cross-Verification (3 questions)

In [ ]:
VERIFICATIONS = [
    ('Which players represent France in volleyball?',
     'SELECT ?p WHERE { ?p wdt:P1532 wd:Q142 . } LIMIT 20', 1),
    ('Which tournaments are organized by the FIVB?',
     'SELECT ?t WHERE { ?t wdt:P664 wd:Q6851 . } LIMIT 20', 17),
    ('Which tournaments are followed by another edition?',
     'SELECT ?t1 ?t2 WHERE { ?t1 wdt:P156 ?t2 . } LIMIT 20', 18),
]

print('Cross-verification — RAG result vs direct SPARQL (ground truth)')
print('='*70)

for label, gold_sparql, qid in VERIFICATIONS:
    print(f'\nQ{qid:02d}: {label}')

    _, gold_rows = run_sparql(gold_sparql)
    enrich_labels([row[0].split('/')[-1] for row in gold_rows])
    print(f'  Ground truth ({len(gold_rows)} results):')
    for row in gold_rows:
        print(f'    - {readable(row[0])}')

    r = next(x for x in eval_results if x['id'] == qid)
    status = '✅ correct' if r['correct'] else '❌ failed'
    print(f'\n  RAG ({r["n_results"]} results) → {status}:')
    for row in r['_rows']:
        print(f'    - {readable(row[0])}')

    if r['baseline'] != '—':
        print(f'\n  Baseline: "{r["baseline"][:250]}"')
        print(f'  → Baseline gives unverifiable answers; RAG returns grounded KB entities.')
    print()

print('='*70)

## 8. Visualisation

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('RAG Evaluation — 20 Questions (Volleyball KB)', fontsize=13, fontweight='bold')

ids    = [r['id']        for r in eval_results]
n_res  = [r['n_results'] for r in eval_results]
colors = ['#4CAF50' if r['correct'] else '#F44336' for r in eval_results]

axes[0].bar(ids, n_res, color=colors)
axes[0].set_title('Results per Question (green=success, red=failed)')
axes[0].set_xlabel('Question #')
axes[0].set_ylabel('Number of results')

n_direct   = sum(1 for r in eval_results if r['correct'] and not r['repairs'])
n_repaired = sum(1 for r in eval_results if r['correct'] and r['repairs'])
n_failed   = sum(1 for r in eval_results if not r['correct'])

axes[1].pie([n_direct, n_repaired, n_failed],
            labels=['Direct success', 'Success after repair', 'Failed'],
            colors=['#4CAF50', '#FF9800', '#F44336'],
            autopct='%1.0f%%', startangle=90)
axes[1].set_title('Query Outcomes')

plt.tight_layout()
plt.savefig(f'{RAG_DIR}/evaluation_chart.png', bbox_inches='tight')
plt.show()
print(f'Chart saved → {RAG_DIR}/evaluation_chart.png')

## 9. Demo

In [ ]:
DEMO_QS = [
    'Which players represent France in volleyball?',
    'Which tournaments are organized by the FIVB?',
    'What playing position does Earvin Ngapeth have?',
]

print('=== Chatbot Demo ===')
print()

for q in DEMO_QS:
    print(f'Question: {q}')
    print()
    print('  Without KB (LLM only):')
    print(f'  {answer_baseline(q).strip()}')
    print()
    print('  With KB (SPARQL + rdflib):')
    r = rag_answer(q)
    if r['rows']:
        enrich_labels([row[0].split('/')[-1] for row in r['rows']])
        for row in r['rows']:
            print(f'    - {readable(row[0])}')
    else:
        print(f'  No results.')
    print(f'  (repairs: {r["repairs"]})')
    print()
    print('-'*55)

# Uncomment to run interactively:
# while True:
#     q = input('Question (or quit): ').strip()
#     if q.lower() == 'quit': break
#     r = rag_answer(q)
#     if r['rows']:
#         enrich_labels([row[0].split('/')[-1] for row in r['rows']])
#         for row in r['rows']:
#             print(f'  - {readable(row[0])}')
#     else:
#         print('No results.')

## 10. Discussion

In [ ]:
n_ok      = sum(1 for r in eval_results if r['correct'])
n_rep     = sum(1 for r in eval_results if r['repairs'] > 0)
n_rep_ok  = sum(1 for r in eval_results if r['repairs'] > 0 and r['correct'])

report = f"""
Lab 4 Report — RAG over RDF/SPARQL
====================================

Setup
-----
Machine : Google Colab (T4 GPU)
LLM     : {MODEL} (Anthropic API)
KB      : expanded_clean.nt ({len(g):,} triples, 51 predicates)
Eval    : {len(EVALUATION_QUESTIONS)} questions

Method
------
We built a schema summary from the KB (prefixes, predicate list, entity URI hints,
sample triples). This summary is injected into the prompt along with few-shot SPARQL
examples to guide the model. If the query returns 0 results or fails, we send the
bad query back to the LLM with the error message and ask it to fix it (up to 2 times).
We also collected baseline answers (LLM without KB) for 5 questions to compare.

Results
-------
RAG success rate : {n_ok}/20 ({n_ok/20*100:.0f}%)
Self-repairs     : {n_rep} triggered, {n_rep_ok} successful

Discussion
----------
Simple lookup questions (players by country, tournament organizer) work well because
the LLM can use the QIDs provided in the hints directly.

Multi-hop questions (e.g. teams from a specific continent) are harder — the model
needs to chain two predicates and doesn't always find the right path.

The baseline comparison shows that the LLM alone gives plausible but unverifiable
answers. RAG returns specific, traceable entities from the KB.

Self-repair helps with syntax errors and wrong predicate names, but is less useful
when the entity simply has no matching triples in the KB.

For a larger-scale deployment, the schema summary should be dynamically adapted per
question instead of being a fixed block sent every time.
"""

print(report)
with open(f'{RAG_DIR}/discussion.txt', 'w') as f:
    f.write(report)
print(f'Saved → {RAG_DIR}/discussion.txt')

print()
print('✅ Lab 4 complete!')
print(f'Files saved to {RAG_DIR}:')
for fname in ['evaluation_table.csv', 'evaluation_chart.png', 'discussion.txt']:
    print(f'  {RAG_DIR}/{fname}')